## 1. Source Files and Initial Schema Inspection

The first step is to inspect the available CSV files and establish their column names, column counts, and basic structural consistency.

This initial inspection reads the headers and one data row from each file without loading the complete datasets into memory. It does not establish the total number of observations, data quality, label distribution, or the suitability of any file for the final analysis. Those properties will be investigated in subsequent steps.

In [1]:
from pathlib import Path
import csv

CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        directory
        for directory in (CURRENT_DIR, *CURRENT_DIR.parents)
        if (directory / "data" / "raw").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Could not locate data/raw from the current directory: {CURRENT_DIR}"
    )

DATA_DIR = PROJECT_ROOT / "data" / "raw"

for filename in ("normal.csv", "attack.csv", "merged.csv"):
    file_path = DATA_DIR / filename

    if not file_path.is_file():
        print(f"\n{filename}: FILE NOT FOUND")
        continue

    with file_path.open("r", newline="", encoding="utf-8-sig") as file:
        reader = csv.reader(file)
        header = next(reader, None)
        first_row = next(reader, None)

    print(f"\n{'=' * 70}")
    print(f"File: {filename}")
    print(f"Size: {file_path.stat().st_size:,} bytes")
    print(f"Number of columns: {len(header) if header else 0}")
    print(f"Header and first-row widths match: "
          f"{header is not None and first_row is not None and len(header) == len(first_row)}")

    print("\nColumn names:")
    if header:
        for index, column in enumerate(header, start=1):
            print(f"{index:>3}. {column}")

    if first_row is not None:
        print("\nFirst five values in the first data row:")
        print(first_row[:5])

        print("\nLast five values in the first data row:")
        print(first_row[-5:])


File: normal.csv
Size: 402,374,633 bytes
Number of columns: 53
Header and first-row widths match: True

Column names:
  1.  Timestamp
  2. FIT101
  3. LIT101
  4.  MV101
  5. P101
  6. P102
  7.  AIT201
  8. AIT202
  9. AIT203
 10. FIT201
 11.  MV201
 12.  P201
 13.  P202
 14. P203
 15.  P204
 16. P205
 17. P206
 18. DPIT301
 19. FIT301
 20. LIT301
 21. MV301
 22. MV302
 23.  MV303
 24. MV304
 25. P301
 26. P302
 27. AIT401
 28. AIT402
 29. FIT401
 30. LIT401
 31. P401
 32. P402
 33. P403
 34. P404
 35. UV401
 36. AIT501
 37. AIT502
 38. AIT503
 39. AIT504
 40. FIT501
 41. FIT502
 42. FIT503
 43. FIT504
 44. P501
 45. P502
 46. PIT501
 47. PIT502
 48. PIT503
 49. FIT601
 50. P601
 51. P602
 52. P603
 53. Normal/Attack

First five values in the first data row:
[' 28/12/2015 10:00:00 AM', '2.427057', '522.8467', '2', '2']

Last five values in the first data row:
['0.000128152', '1', '1', '1', 'Normal']

File: attack.csv
Size: 15,162,438 bytes
Number of columns: 53
Header and first-row w

## 2. Structural and Temporal Overview

This step examines the complete set of source files to establish their record counts, normalized schemas, timestamp coverage, chronological ordering, and label distributions.

The files will be processed in chunks to limit memory usage. Column names will be normalized in memory for schema comparison; the original files will remain unchanged.

The results will determine whether the files are structurally consistent and provide the evidence needed to select the appropriate source for subsequent analysis.

In [ ]:
import pandas as pd
import numpy as np

# Locate the project data directory without hardcoding the machine-specific path.
current_dir = Path.cwd().resolve()

project_root = next(
    (
        directory
        for directory in (current_dir, *current_dir.parents)
        if (directory / "data" / "raw").is_dir()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        f"Could not locate data/raw from the current directory: {current_dir}"
    )

data_dir = project_root / "data" / "raw"
filenames = ("normal.csv", "attack.csv", "merged.csv")

schemas = {}
results = {}

for filename in filenames:
    file_path = data_dir / filename

    if not file_path.is_file():
        raise FileNotFoundError(f"Required file not found: {file_path}")

    raw_header = pd.read_csv(
        file_path,
        nrows=0,
        encoding="utf-8-sig",
    ).columns.tolist()

    normalized_header = [str(column).strip() for column in raw_header]

    schemas[filename] = normalized_header

    rows = 0
    invalid_timestamps = 0
    label_counts = {}

    minimum_timestamp = None
    maximum_timestamp = None
    previous_timestamp = None
    chronological = True

    whitespace_headers = [
        repr(column)
        for column in raw_header
        if str(column) != str(column).strip()
    ]

    duplicate_normalized_headers = (
        len(normalized_header) != len(set(normalized_header))
    )

    for chunk in pd.read_csv(
        file_path,
        chunksize=100_000,
        encoding="utf-8-sig",
        skipinitialspace=True,
    ):
        chunk.columns = [str(column).strip() for column in chunk.columns]

        if "Timestamp" not in chunk.columns:
            raise ValueError(f"{filename}: Timestamp column is missing.")

        if "Normal/Attack" not in chunk.columns:
            raise ValueError(f"{filename}: Normal/Attack label is missing.")

        rows += len(chunk)

        # Parse timestamps using the format observed in the initial inspection.
        timestamps = pd.to_datetime(
            chunk["Timestamp"].astype("string").str.strip(),
            format="%d/%m/%Y %I:%M:%S %p",
            errors="coerce",
        )

        invalid_timestamps += int(timestamps.isna().sum())

        valid_timestamps = timestamps.dropna()

        if not valid_timestamps.empty:
            chunk_minimum = valid_timestamps.min()
            chunk_maximum = valid_timestamps.max()

            minimum_timestamp = (
                chunk_minimum
                if minimum_timestamp is None
                else min(minimum_timestamp, chunk_minimum)
            )

            maximum_timestamp = (
                chunk_maximum
                if maximum_timestamp is None
                else max(maximum_timestamp, chunk_maximum)
            )

            if not valid_timestamps.is_monotonic_increasing:
                chronological = False

            if (
                previous_timestamp is not None
                and valid_timestamps.iloc[0] < previous_timestamp
            ):
                chronological = False

            previous_timestamp = valid_timestamps.iloc[-1]

        labels = chunk["Normal/Attack"].astype("string").str.strip()

        for label, count in labels.value_counts(dropna=False).items():
            label_name = "<MISSING>" if pd.isna(label) else str(label)
            label_counts[label_name] = (
                label_counts.get(label_name, 0) + int(count)
            )

    results[filename] = {
        "rows": rows,
        "columns": len(normalized_header),
        "invalid_timestamps": invalid_timestamps,
        "minimum_timestamp": minimum_timestamp,
        "maximum_timestamp": maximum_timestamp,
        "chronological": chronological,
        "label_counts": label_counts,
        "whitespace_headers": whitespace_headers,
        "duplicate_normalized_headers": duplicate_normalized_headers,
    }

    print(f"\n{'=' * 70}")
    print(f"FILE: {filename}")
    print(f"Rows: {rows:,}")
    print(f"Columns: {len(normalized_header)}")
    print(f"Invalid timestamps: {invalid_timestamps:,}")
    print(f"Earliest timestamp: {minimum_timestamp}")
    print(f"Latest timestamp: {maximum_timestamp}")
    print(f"Timestamps chronologically ordered: {chronological}")
    print(f"Label counts: {label_counts}")
    print(f"Headers with surrounding whitespace: {whitespace_headers}")
    print(
        "Duplicate column names after normalization: "
        f"{duplicate_normalized_headers}"
    )

print(f"\n{'=' * 70}")
print("SCHEMA COMPARISON")

reference_schema = schemas["normal.csv"]

for filename, schema in schemas.items():
    print(
        f"{filename} matches normal.csv after trimming header whitespace: "
        f"{schema == reference_schema}"
    )


FILE: normal.csv
Rows: 1,387,098
Columns: 53
Invalid timestamps: 0
Earliest timestamp: 2015-12-22 16:00:00
Latest timestamp: 2016-01-02 14:59:59
Timestamps chronologically ordered: False
Label counts: {'Normal': 1387098}
Headers with surrounding whitespace: ["' Timestamp'", "' MV101'", "' AIT201'", "' MV201'", "' P201'", "' P202'", "' P204'", "' MV303'"]
Duplicate column names after normalization: False

FILE: attack.csv
Rows: 54,621
Columns: 53
Invalid timestamps: 0
Earliest timestamp: 2015-12-28 10:29:14
Latest timestamp: 2016-01-02 13:41:11
Timestamps chronologically ordered: True
Label counts: {'Attack': 54621}
Headers with surrounding whitespace: ["' Timestamp'", "' MV101'", "' AIT201'", "' MV201'", "' P201'", "' P202'", "' P204'", "' MV303'"]
Duplicate column names after normalization: False

FILE: merged.csv
Rows: 1,441,719
Columns: 53
Invalid timestamps: 0
Earliest timestamp: 2015-12-22 16:00:00
Latest timestamp: 2016-01-02 14:59:59
Timestamps chronologically ordered: False
La

## 3. Investigation of Chronological Ordering

The initial inspection found that `normal.csv` and `merged.csv` are not chronologically ordered, although all timestamps were successfully parsed using the observed timestamp format.

This step identifies backward timestamp transitions in the original row order, including transitions across chunk boundaries. It also counts adjacent rows with identical timestamps.

The purpose is to locate and quantify the ordering violations before deciding how the data should be prepared for temporal analysis.

The original CSV files will remain unchanged. No rows will be reordered or removed during this investigation.

In [ ]:
# Locate the project directory without a machine-specific absolute path.
current_dir = Path.cwd().resolve()

project_root = next(
    (
        directory
        for directory in (current_dir, *current_dir.parents)
        if (directory / "data" / "raw").is_dir()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        f"Could not locate data/raw from: {current_dir}"
    )

data_dir = project_root / "data" / "raw"

files_to_check = ("normal.csv", "merged.csv")
chunk_size = 100_000
timestamp_format = "%d/%m/%Y %I:%M:%S %p"
max_examples = 10


for filename in files_to_check:
    file_path = data_dir / filename

    if not file_path.is_file():
        raise FileNotFoundError(f"File not found: {file_path}")

    rows_scanned = 0
    invalid_timestamp_count = 0

    backward_within_chunk = 0
    backward_cross_chunk = 0

    duplicate_within_chunk = 0
    duplicate_cross_chunk = 0

    backward_examples = []
    duplicate_examples = []
    invalid_examples = []

    largest_backward = None

    minimum_timestamp = None
    maximum_timestamp = None
    previous_timestamp = None

    for chunk in pd.read_csv(
        file_path,
        chunksize=chunk_size,
        encoding="utf-8-sig",
        skipinitialspace=True,
    ):
        # Use local positional indices for every chunk.
        chunk = chunk.reset_index(drop=True)
        chunk.columns = [
            str(column).strip()
            for column in chunk.columns
        ]

        if "Timestamp" not in chunk.columns:
            raise ValueError(
                f"{filename}: Timestamp column is missing."
            )

        timestamps = pd.to_datetime(
            chunk["Timestamp"].astype("string").str.strip(),
            format=timestamp_format,
            errors="coerce",
        )

        if timestamps.empty:
            continue

        # Record invalid timestamps and their original CSV line numbers.
        invalid_positions = np.flatnonzero(
            timestamps.isna().to_numpy()
        )

        invalid_timestamp_count += len(invalid_positions)

        for position in invalid_positions:
            if len(invalid_examples) >= max_examples:
                break

            invalid_examples.append(
                rows_scanned + int(position) + 2
            )

        # Track timestamp coverage.
        valid_timestamps = timestamps.dropna()

        if not valid_timestamps.empty:
            chunk_minimum = valid_timestamps.min()
            chunk_maximum = valid_timestamps.max()

            minimum_timestamp = (
                chunk_minimum
                if minimum_timestamp is None
                else min(minimum_timestamp, chunk_minimum)
            )

            maximum_timestamp = (
                chunk_maximum
                if maximum_timestamp is None
                else max(maximum_timestamp, chunk_maximum)
            )

        # Check the boundary between consecutive chunks.
        first_timestamp = timestamps.iloc[0]

        if (
            rows_scanned > 0
            and pd.notna(previous_timestamp)
            and pd.notna(first_timestamp)
        ):
            previous_line = rows_scanned + 1
            current_line = rows_scanned + 2

            if first_timestamp < previous_timestamp:
                backward_cross_chunk += 1

                delta_seconds = (
                    previous_timestamp - first_timestamp
                ).total_seconds()

                transition = {
                    "location": "cross_chunk",
                    "previous_csv_line": previous_line,
                    "current_csv_line": current_line,
                    "previous_timestamp": previous_timestamp,
                    "current_timestamp": first_timestamp,
                    "backward_seconds": delta_seconds,
                }

                if len(backward_examples) < max_examples:
                    backward_examples.append(transition)

                if (
                    largest_backward is None
                    or delta_seconds
                    > largest_backward["backward_seconds"]
                ):
                    largest_backward = transition.copy()

            elif first_timestamp == previous_timestamp:
                duplicate_cross_chunk += 1

                if len(duplicate_examples) < max_examples:
                    duplicate_examples.append(
                        {
                            "location": "cross_chunk",
                            "previous_csv_line": previous_line,
                            "current_csv_line": current_line,
                            "timestamp": first_timestamp,
                        }
                    )

        # Compare adjacent timestamps within the current chunk.
        differences = timestamps.diff()

        backward_positions = np.flatnonzero(
            (differences < pd.Timedelta(0)).to_numpy()
        )

        duplicate_positions = np.flatnonzero(
            (differences == pd.Timedelta(0)).to_numpy()
        )

        for position in backward_positions:
            position = int(position)

            previous_value = timestamps.iloc[position - 1]
            current_value = timestamps.iloc[position]

            delta_seconds = (
                previous_value - current_value
            ).total_seconds()

            backward_within_chunk += 1

            transition = {
                "location": "within_chunk",
                "previous_csv_line": rows_scanned + position + 1,
                "current_csv_line": rows_scanned + position + 2,
                "previous_timestamp": previous_value,
                "current_timestamp": current_value,
                "backward_seconds": delta_seconds,
            }

            if len(backward_examples) < max_examples:
                backward_examples.append(transition)

            if (
                largest_backward is None
                or delta_seconds
                > largest_backward["backward_seconds"]
            ):
                largest_backward = transition.copy()

        for position in duplicate_positions:
            position = int(position)

            duplicate_within_chunk += 1

            if len(duplicate_examples) < max_examples:
                duplicate_examples.append(
                    {
                        "location": "within_chunk",
                        "previous_csv_line": rows_scanned + position + 1,
                        "current_csv_line": rows_scanned + position + 2,
                        "timestamp": timestamps.iloc[position],
                    }
                )

        previous_timestamp = timestamps.iloc[-1]
        rows_scanned += len(chunk)

    total_backward = (
        backward_within_chunk + backward_cross_chunk
    )

    total_adjacent_duplicates = (
        duplicate_within_chunk + duplicate_cross_chunk
    )

    print("\n" + "=" * 75)
    print(f"FILE: {filename}")
    print(f"Rows scanned: {rows_scanned:,}")
    print(f"Invalid timestamps: {invalid_timestamp_count:,}")
    print(f"Earliest valid timestamp: {minimum_timestamp}")
    print(f"Latest valid timestamp: {maximum_timestamp}")

    print("\nBackward timestamp transitions")
    print(f"  Within chunks: {backward_within_chunk:,}")
    print(f"  Across chunks: {backward_cross_chunk:,}")
    print(f"  Total: {total_backward:,}")

    print("\nAdjacent identical timestamp pairs")
    print(f"  Within chunks: {duplicate_within_chunk:,}")
    print(f"  Across chunks: {duplicate_cross_chunk:,}")
    print(f"  Total: {total_adjacent_duplicates:,}")

    print("\nFirst backward-transition examples")

    if backward_examples:
        print(
            pd.DataFrame(backward_examples).to_string(
                index=False
            )
        )
    else:
        print("No backward transitions found.")

    print("\nLargest backward transition")

    if largest_backward is not None:
        print(pd.Series(largest_backward).to_string())
    else:
        print("No backward transitions found.")

    print("\nFirst adjacent duplicate timestamp examples")

    if duplicate_examples:
        print(
            pd.DataFrame(duplicate_examples).to_string(
                index=False
            )
        )
    else:
        print("No adjacent duplicate timestamp pairs found.")

    print("\nFirst invalid-timestamp CSV line numbers")

    if invalid_examples:
        print(invalid_examples)
    else:
        print("No invalid timestamps found.")

    if invalid_timestamp_count > 0:
        print(
            "\nNOTE: Timestamp ordering around rows with invalid "
            "timestamps cannot be fully established by adjacent-row checks."
        )


FILE: normal.csv
Rows scanned: 1,387,098
Invalid timestamps: 0
Earliest valid timestamp: 2015-12-22 16:00:00
Latest valid timestamp: 2016-01-02 14:59:59

Backward timestamp transitions
  Within chunks: 2
  Across chunks: 0
  Total: 2

Adjacent identical timestamp pairs
  Within chunks: 0
  Across chunks: 0
  Total: 0

First backward-transition examples
    location  previous_csv_line  current_csv_line  previous_timestamp   current_timestamp  backward_seconds
within_chunk             395299            395300 2016-01-02 14:59:59 2015-12-22 16:00:00          946799.0
within_chunk             892099            892100 2015-12-28 09:59:59 2015-12-22 16:30:00          494999.0

Largest backward transition
location                     within_chunk
previous_csv_line                  395299
current_csv_line                   395300
previous_timestamp    2016-01-02 14:59:59
current_timestamp     2015-12-22 16:00:00
backward_seconds                 946799.0

First adjacent duplicate timestamp exa

In [ ]:
# Locate data/raw without a machine-specific path.
raw_dir = next(
    parent / "data" / "raw"
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "data" / "raw").is_dir()
)

files_to_inspect = ("normal.csv", "merged.csv")

# Transition lines reported by the previous diagnostic (1-based, header = line 1).
# These are the CSV lines at which the timestamp jumps backward.
normal_transitions = (395300, 892100)

timestamp_format = "%d/%m/%Y %I:%M:%S %p"
chunk_size = 200_000

for filename in files_to_inspect:
    file_path = raw_dir / filename

    # --- Pass 1: compute unique timestamps and adjacent-duplicate count. ---
    # Streaming over only the Timestamp column keeps memory bounded.
    timestamp_parts = []
    for chunk in pd.read_csv(
        file_path,
        usecols=["Timestamp"],
        chunksize=chunk_size,
        encoding="utf-8-sig",
        skipinitialspace=True,
    ):
        chunk.columns = [str(c).strip() for c in chunk.columns]
        parsed = pd.to_datetime(
            chunk["Timestamp"].astype("string").str.strip(),
            format=timestamp_format,
            errors="coerce",
        )
        timestamp_parts.append(parsed)

    timestamps = pd.concat(timestamp_parts, ignore_index=True)
    n_rows = len(timestamps)
    n_invalid = int(timestamps.isna().sum())
    n_unique = int(timestamps.dropna().nunique())
    n_non_adjacent_duplicates = n_rows - n_invalid - n_unique

    # --- Pass 2: segment boundaries from the known transition lines. ---
    # For normal.csv, two transitions produce three segments.
    # For merged.csv, the same two transitions produce the same three
    # segments followed by the appended attack block.
    if filename == "normal.csv":
        segment_starts = [1] + list(normal_transitions)
    else:  # merged.csv: same internal structure, plus the trailing attack block.
        segment_starts = [1] + list(normal_transitions)

    segment_ends = [s - 1 for s in segment_starts[1:]] + [n_rows]

    print(f"\n{'=' * 80}")
    print(f"FILE: {filename}")
    print("=" * 80)
    print(f"Rows (excluding header): {n_rows:,}")
    print(f"Invalid timestamps:      {n_invalid:,}")
    print(f"Unique timestamps:       {n_unique:,}")
    print(f"Non-adjacent duplicate timestamps (n_rows - n_unique): "
          f"{n_non_adjacent_duplicates:,}")
    print()
    print("Segments (1-based CSV lines, where header = line 1):")

    for index, (start, end) in enumerate(zip(segment_starts, segment_ends), start=1):
        segment = timestamps.iloc[start - 1:end]
        segment_first = segment.dropna().iloc[0]
        segment_last = segment.dropna().iloc[-1]
        print(
            f"  Segment {index}: rows {start:>10,} .. {end:>10,}  "
            f"({len(segment):>10,} rows)  "
            f"timestamps {segment_first} .. {segment_last}"
        )

    # Cross-segment timestamp overlap: does any timestamp appear in more than
    # one segment? This directly tests whether segments B and C overlap in time.
    segment_sets = [
        set(timestamps.iloc[start - 1:end].dropna())
        for start, end in zip(segment_starts, segment_ends)
    ]
    if len(segment_sets) >= 2:
        print()
        print("Pairwise segment timestamp overlap (count of shared timestamps):")
        for i in range(len(segment_sets)):
            for j in range(i + 1, len(segment_sets)):
                shared = segment_sets[i] & segment_sets[j]
                print(f"  Segment {i + 1} ∩ Segment {j + 1}: {len(shared):,}")


FILE: normal.csv
Rows (excluding header): 1,387,098
Invalid timestamps:      0
Unique timestamps:       892,098
Non-adjacent duplicate timestamps (n_rows - n_unique): 495,000

Segments (1-based CSV lines, where header = line 1):
  Segment 1: rows          1 ..    395,299  (   395,299 rows)  timestamps 2015-12-28 10:00:00 .. 2015-12-22 16:00:00
  Segment 2: rows    395,300 ..    892,099  (   496,800 rows)  timestamps 2015-12-22 16:00:01 .. 2015-12-22 16:30:00
  Segment 3: rows    892,100 ..  1,387,098  (   494,999 rows)  timestamps 2015-12-22 16:30:01 .. 2015-12-28 09:59:59

Pairwise segment timestamp overlap (count of shared timestamps):
  Segment 1 ∩ Segment 2: 0
  Segment 1 ∩ Segment 3: 0
  Segment 2 ∩ Segment 3: 494,999

FILE: merged.csv
Rows (excluding header): 1,441,719
Invalid timestamps:      0
Unique timestamps:       946,719
Non-adjacent duplicate timestamps (n_rows - n_unique): 495,000

Segments (1-based CSV lines, where header = line 1):
  Segment 1: rows          1 ..    3

In [ ]:
from PIL import GimpGradientFile
FILES = ("normal.csv", "attack.csv", "merged.csv")
CHUNK_SIZE = 100_000

reconciliation = {}

for filename in FILES:
    file_path = DATA_DIR / filename